<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l7.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L7 · Walk-forward
6 ventanas (entrena 50 días, prueba 15): la ganadora de cada tramo se evalúa en datos que no vio.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c2_l7_ventanas.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/python/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
df["robusta"] = (df["sharpe_oos"] > 0) & (df["gap"].abs() < 1.5)
print(df[["ventana", "mejor_fast", "mejor_lento", "sharpe_is", "sharpe_oos", "gap"]].to_string(index=False))
print(f"\nventanas robustas: {int(df['robusta'].sum())} de {len(df)}  |  gap medio: {df['gap'].mean():+.3f}")

## La ganadora cambia, el gap manda
Si la mejor pareja (fast,slow) rota entre ventanas, no hay parámetro mágico: hay regímenes. El gap IS−OOS lo delata.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df["sharpe_is"], df["sharpe_oos"], c=["#5eead4" if r else "#f59e0b" for r in df["robusta"]], s=80)
for _, r in df.iterrows():
    ax.text(r["sharpe_is"], r["sharpe_oos"], f"W{int(r['ventana'])}", fontsize=8)
lims = [df[["sharpe_is", "sharpe_oos"]].min().min()-0.3, df[["sharpe_is", "sharpe_oos"]].max().max()+0.3]
ax.plot(lims, lims, "--", c="#52525b", label="diagonal (gap=0)")
ax.set_xlabel("Sharpe IS")
ax.set_ylabel("Sharpe OOS")
ax.set_title("Walk-forward: cada ventana es un examen nuevo")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Chequeos automáticos
assert len(df) == 6, "se esperan 6 ventanas"
assert (df["fin_test"] - df["ini_test"] == 15).all(), "cada test cubre 15 días"
assert abs(df["gap"] - (df["sharpe_is"] - df["sharpe_oos"])).max() < 0.002, "gap = IS − OOS"
assert int(df['robusta'].sum()) == 1, "nº de ventanas robustas determinista"
print("OK: walk-forward verificado")